# GRUPO 43 — Entrega 6: diagnóstico del techo y *stacking* refinado

**Objetivo:** subir el F1 macro (y la accuracy de Kaggle) del *stacking* de la entrega 5 sin salir de las reglas de la Parte 1. La entrega 5 obtuvo 0.915 de F1 en nuestra partición de validación, pero entre 0.89 y 0.899 en Kaggle. Antes de cambiar el modelo, esta entrega responde **por qué** y **cuánto margen queda**.

### Qué cambia respecto a la entrega 5

1. **Diagnóstico del techo (sección 3).** El conjunto de datos tiene dos familias de opiniones. En la de **adjetivos** (*"el motor se mantuvo resistente"*) la última opinión decide siempre. En la de **frases hechas** (*"tiré el dinero"*, *"fue todo un acierto"*) las reseñas mixtas tienen una etiqueta que se comporta como una moneda al aire. Ese grupo pone un techo cercano a 0.92 que ningún modelo puede superar.
2. **Validación cruzada (sección 7).** La partición 80/20 con semilla 42 resultó optimista. Para comparar modelos usamos predicciones *out-of-fold* (OOF) de 5 *folds* sobre las 12.000 reseñas, y miramos aparte el acierto en las reseñas que sí se pueden predecir.
3. **Normalización de letras repetidas** en `limpiar_texto`. Corrige los errores de tipeo del conjunto de datos (*conffiable*, *esttrella*, *arrrepiento*).
4. **Nuevas variables de polaridad** (`PolaridadClausulas2`):
   - léxico con bigramas (*poco fiable*, *mal terminada*, *de mala calidad*);
   - palabra más polar de cada cláusula;
   - "última cláusula fuertemente polar", que es robusta al relleno que viene después de la opinión;
   - indicador de cierre ambivalente.

### Reglas del proyecto (Parte 1)

| Regla | Cómo se cumple |
|---|---|
| Sin aprendizaje profundo, transformers ni *embeddings* preentrenados | Todo sale de conteos de palabras: BoW, TF‑IDF y n‑gramas. |
| Solo clasificadores de scikit-learn | `LogisticRegression`, `MultinomialNB` y `RandomForestClassifier`. Las variables y el ensamble usan `CountVectorizer`, `cross_val_predict`, `StratifiedKFold` y `Pipeline`. |
| `eval.csv` no se usa para entrenar ni para escoger modelos | Solo se usa para predecir (sección 8). En la sección 3.4 solo se **cuentan** frases de su texto para describir el conjunto. |
| Resultados replicables | Semillas fijas, todo el código en este notebook y modelo guardado con `joblib`. |

## 1. Preparación del entorno y datos

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import re, unicodedata
from collections import Counter
from contextlib import contextmanager
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42
ORDER = ["negativo", "neutral", "positivo"]
COLORS = {"negativo": "#E45756", "neutral": "#9D9D9D", "positivo": "#54A24B"}
DATA_DIR = Path("../data")
Path("models").mkdir(exist_ok=True)

train_df = pd.read_csv(DATA_DIR / "train.csv")
eval_df = pd.read_csv(DATA_DIR / "eval.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert eval_df["id"].equals(sample_sub["id"])
print(train_df.shape, eval_df.shape)

### 1.1 División train / validación

Usamos la misma partición estratificada 80/20 de todas las entregas, para poder comparar con ellas. **Control:** la proporción de clases debe ser igual en ambos conjuntos.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train_df["text"], train_df["label"], test_size=0.2, stratify=train_df["label"], random_state=RANDOM_STATE)

prop = pd.DataFrame({"train": y_train.value_counts(normalize=True),
                     "validación": y_val.value_counts(normalize=True)}).reindex(ORDER)
ax = prop.plot(kind="bar", rot=0, figsize=(6, 3.2), color=["#4C78A8", "#F58518"])
ax.set_title("Proporción de clases: train vs validación"); ax.set_ylabel("proporción")
plt.tight_layout(); plt.show()
prop.round(3)

**Qué sale:** proporciones idénticas (≈ 35/30/35). La partición es comparable.

## 2. Preparación del texto

Reutilizamos las funciones de la entrega 5 con un cambio y una lista nueva:

- `limpiar_texto`: minúsculas, sin tildes y solo letras. **Nuevo:** colapsa letras repetidas (`conffiable` → `confiable`). El conjunto de datos trae errores de tipeo inyectados que duplican letras, y cada variante era una palabra distinta para BoW y TF‑IDF. La regla se aplica igual a todas las palabras, así que también cambia algunas palabras correctas (`llegó` → `lego`, `estrella` → `estrela`). Como el cambio es el mismo en train y en eval, no se pierde información. La bandera `NORMALIZAR_REPETIDAS` permite reproducir la limpieza de la entrega 5 para comparar.
- `separar_clausulas`, `CONECTORES` y `conector`: sin cambios. Se corta en `. ! ? ;` y antes de *pero*, *aunque* y *sin embargo*.
- **`CIERRES` (nuevo):** frases con las que terminan muchas reseñas y que no expresan una opinión (*"cada quien que juzgue"*, *"ni fu ni fa"*). Las encontramos igual que los conectores, revisando las últimas cláusulas más frecuentes de `train.csv` (sección 3.1).

In [ ]:
NORMALIZAR_REPETIDAS = True

def quitar_tildes(t):
    t = unicodedata.normalize("NFKD", t)
    return "".join(c for c in t if not unicodedata.combining(c))

def limpiar_texto(t):
    t = quitar_tildes(str(t).lower())
    t = re.sub(r"[^a-zñ\s]", " ", t)
    if NORMALIZAR_REPETIDAS:
        t = re.sub(r"([a-zñ])\1+", r"\1", t)   # "conffiable" -> "confiable"
    return re.sub(r"\s+", " ", t).strip()

@contextmanager
def limpieza_entrega5():
    # Reproduce la limpieza de la entrega 5 (sin colapsar letras) dentro de un bloque `with`.
    global NORMALIZAR_REPETIDAS
    NORMALIZAR_REPETIDAS = False
    try:
        yield
    finally:
        NORMALIZAR_REPETIDAS = True

def separar_clausulas(t):
    t = re.sub(r"\b(pero|aunque|sin embargo)\b", r". \1", str(t), flags=re.I)
    c = [limpiar_texto(p) for p in re.split(r"[.!?;]+", t)]
    c = [x for x in c if x]
    return c if c else [""]

CONECTORES = ["eso si", "aun asi", "por otro lado", "la verdad", "por cierto", "de paso", "con todo", "ademas",
              "al final", "a fin de cuentas", "ahora", "pero", "aunque", "sin embargo", "en fin", "para mi",
              "en mi opinion", "para ser sincero", "la verdad sea dicha", "hay de todo", "para gustos", "sigo con dudas"]
_ORD = sorted(CONECTORES, key=len, reverse=True)

def conector(c):
    for k in _ORD:
        if c == k or c.startswith(k + " "):
            return CONECTORES.index(k)
    return -1

CIERRES = ["cada quien", "para gustos", "sigo con dudas", "ni fu ni fa", "a criterio", "hay de todo", "no sabria bien",
           "sopesalo", "no me decido", "ni lo mejor ni lo peor", "ni tan bueno ni tan malo", "ahi lo dejo",
           "sentimientos encontrados"]

def es_cierre(c):
    return any(k in c for k in CIERRES)

### 2.1 Control: ¿qué corrige la normalización de letras repetidas?

Contamos el vocabulario con y sin la normalización y listamos palabras **raras** (1 o 2 apariciones) con letras repetidas cuya versión normalizada es una palabra **frecuente** (20 apariciones o más). Esas son los errores de tipeo que ahora el modelo reconoce.

In [ ]:
with limpieza_entrega5():
    voc_sin = Counter(w for t in train_df["text"] for w in limpiar_texto(t).split())
voc_con = Counter(w for t in train_df["text"] for w in limpiar_texto(t).split())
colapsar = lambda w: re.sub(r"([a-zñ])\1+", r"\1", w)

corregidas = sorted(((w, colapsar(w)) for w, n in voc_sin.items()
                     if n <= 2 and re.search(r"([a-zñ])\1", w) and voc_con.get(colapsar(w), 0) >= 20),
                    key=lambda p: -voc_con[p[1]])
print(f"Vocabulario: {len(voc_sin):,} palabras sin normalizar -> {len(voc_con):,} normalizado")
print(f"Tipeos raros que pasan a una palabra frecuente: {len(corregidas)}")
pd.DataFrame(corregidas[:15], columns=["palabra original", "normalizada"]).T

In [ ]:
ejemplo = train_df.loc[train_df["text"].str.contains(", pero"), "text"].iloc[0]
print(ejemplo, "\n")
for c in separar_clausulas(ejemplo):
    k = conector(c)
    print(f"  [{CONECTORES[k] if k >= 0 else '-':>14}]  {c}")

**Qué sale:** el vocabulario se reduce y cientos de variantes con letras duplicadas (*conffiable*, *esttrella*, *arrrepiento*…) vuelven a su palabra base. El separador de cláusulas funciona igual que en la entrega 5.

## 3. Diagnóstico: ¿por qué no pasamos de ~0.91?

En las entregas 4 y 5 vimos que *la opinión final manda*, pero también que había reseñas que contradecían esa regla. Aquí medimos **cuántas** son y si existe alguna forma de predecirlas.

Para medirlo construimos dos listas a partir de las frases de opinión más frecuentes de `train.csv`:

- **adjetivos** de opinión que siguen a un verbo como *se mantuvo*, *luce*, *me resultó* o *se sintió* (*resistente*, *poco fiable*, *de mala calidad*…);
- **frases hechas** (*tiré el dinero*, *fue todo un acierto*, *me decepcionó*…).

**Estas listas solo se usan en esta sección de diagnóstico, no como variables del modelo.** El modelo sigue construido sobre BoW, TF‑IDF y n‑gramas.

In [ ]:
_norm = lambda xs: [limpiar_texto(x) for x in xs]
ADJ_POS = set(_norm(
    "resistente eficiente impecable confiable funcional veloz sobresaliente potente decente superior notable formidable "
    "excelente espectacular durable agradable cumplidor cumplidora practica practico buenisima buenisimo rapidisima "
    "rapidisimo comoda comodo comodisima comodisimo buena bueno redonda redondo optimo optima precisa preciso correcta "
    "correcto ligera ligero solida solido robusta robusto estupenda estupendo completa completo nitida nitido logrado "
    "lograda fiable".split()))
ADJ_NEG = set(_norm(
    "terrible decepcionante torpe pobre debil deficiente corriente endeble mediocre desagradable inestable fragil "
    "inconsistente olvidable incumplidor incumplidora ruidosa ruidoso delicada delicado imprecisa impreciso lentisima "
    "lentisimo chapucera chapucero pesima pesimo malisima malisimo ordinaria ordinario floja flojo incomoda incomodo "
    "defectuosa defectuoso limitada limitado aparatosa aparatoso basica basico".split()))
FRASES_NEG = _norm(["tire el dinero", "decepciono", "error de compra", "ni regalado", "arrepiento", "arrepentido",
                    "una estrella", "una sola estrella", "dolor de cabeza", "harto", "muy molesto", "defraudo todas",
                    "no vale lo que cuesta", "para nada lo que esperaba", "queda muy corto", "queda corto",
                    "tener problemas", "debajo de la media", "no me convence", "recomiendo a nadie", "no me gusto",
                    "dano antes", "funciona bastante mal", "no me termina de gustar"])
FRASES_POS = _norm(["todas mis expectativas", "todo un acierto", "valio la pena", "vale totalmente la pena",
                    "de maravilla", "me encanto", "feliz", "ojos cerrados", "cinco estrellas", "ni una sola queja",
                    "me alegra", "tal como esperaba", "de sobra", "encantado", "vida mas facil", "encima de la media",
                    "lo que estaba buscando", "sin pensarlo", "para bien", "ninguna queja", "cero quejas", "contento"])
VERBO = r"(?:se mantuvo|luce|se ve|me resulto ser|me resulto|resulto ser|resulto|termino siendo|se sintio|me parecio|quedo|salio|me salio|es|esta|fue)"
RX_ADJ = re.compile(VERBO + r"\s+((?:muy|bastante|algo|un poco|demasiado|poco|mal|bien)\s+|de (?:mala|buena) )?([a-zñ]+)")

def signo_adjetivo(c):
    s = 0
    for m in RX_ADJ.finditer(c):
        mod, w = (m.group(1) or "").strip(), m.group(2)
        if mod in ("de mala", "de buena") and w == "calidad":
            s += 1 if mod == "de buena" else -1
        elif w in ADJ_POS:
            s += -1 if mod in ("poco", "mal") else 1
        elif w in ADJ_NEG:
            s -= 1
        elif w in ("hecha", "hecho", "terminada", "terminado", "construida", "construido") and mod in ("mal", "bien"):
            s += 1 if mod == "bien" else -1
    return int(np.sign(s))

def signo_frase(c):
    return int(np.sign(sum(k in c for k in FRASES_POS) - sum(k in c for k in FRASES_NEG)))

def es_mixta_frases(texto):
    # Reseña con al menos dos frases hechas de polaridad opuesta (no usa la etiqueta).
    return len({signo_frase(c) for c in separar_clausulas(texto)} - {0}) > 1

def grupo(texto, etiqueta):
    if etiqueta == "neutral":
        return "neutral"
    return "mixta frases hechas" if es_mixta_frases(texto) else "determinista"

### 3.1 Cierres ambivalentes

Listamos las últimas cláusulas más frecuentes de `train.csv` y la clase de las reseñas que terminan con ellas.

In [ ]:
ultimas = train_df["text"].map(lambda t: separar_clausulas(t)[-1])
top = ultimas.value_counts().head(18).index
tabla_cierres = pd.crosstab(ultimas[ultimas.isin(top)], train_df["label"]).reindex(top)[ORDER]
tabla_cierres["¿en CIERRES?"] = [es_cierre(c) for c in tabla_cierres.index]
fin_cierre = ultimas.map(es_cierre)
print(f"Reseñas que terminan en un cierre ambivalente: {fin_cierre.sum()} ({fin_cierre.mean():.1%})")
print(train_df.loc[fin_cierre, "label"].value_counts().reindex(ORDER).to_dict())
tabla_cierres

**Qué sale:** las frases de cierre más comunes (*cada quien que juzgue*, *para gustos colores*, *ni fu ni fa*…) **nunca** aparecen en reseñas neutrales y se reparten casi por mitades entre positivas y negativas. Señalan una reseña mixta, pero no dicen hacia qué lado se inclina. Los cierres *nada más que agregar / comentar* son de reseñas neutrales y no están en la lista.

### 3.2 Dos familias de opiniones

Tomamos las reseñas no neutrales que tienen **dos opiniones de polaridad opuesta** y medimos con qué frecuencia la etiqueta coincide con la **última** opinión. Separamos según el tipo de opinión: solo adjetivos o solo frases hechas.

In [ ]:
filas = []
for texto, etiqueta in zip(train_df["text"], train_df["label"]):
    if etiqueta == "neutral":
        continue
    cl = separar_clausulas(texto)
    sa = [(i, signo_adjetivo(c)) for i, c in enumerate(cl) if signo_adjetivo(c) != 0]
    sf = [(i, signo_frase(c)) for i, c in enumerate(cl) if signo_frase(c) != 0]
    if len({s for _, s in sa}) > 1 and not sf:
        tipo, (i, s) = "adjetivos", sa[-1]
    elif len({s for _, s in sf}) > 1 and not sa:
        tipo, (i, s) = "frases hechas", sf[-1]
    else:
        continue
    k = conector(cl[i])
    filas.append({"tipo": tipo, "gana_ultima": s == (1 if etiqueta == "positivo" else -1),
                  "conector": CONECTORES[k] if k >= 0 else "(ninguno)"})
mixtas = pd.DataFrame(filas)
resumen_tipo = mixtas.groupby("tipo")["gana_ultima"].agg(["mean", "size"]).rename(columns={"mean": "% gana la última", "size": "reseñas"})

top_con = mixtas["conector"].value_counts().head(8).index
por_con = mixtas[mixtas["conector"].isin(top_con)].pivot_table(index="conector", columns="tipo", values="gana_ultima", aggfunc="mean").reindex(top_con)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1, 2]})
resumen_tipo["% gana la última"].plot(kind="bar", rot=0, ax=axes[0], color=["#4C78A8", "#F58518"])
axes[0].axhline(0.5, ls="--", color="gray"); axes[0].set_ylim(0, 1.05)
axes[0].set_title("Reseñas con dos opiniones opuestas:\n¿la etiqueta coincide con la última?")
for i, v in enumerate(resumen_tipo["% gana la última"]):
    axes[0].text(i, v + 0.02, f"{v:.0%}", ha="center")
por_con.plot(kind="bar", rot=30, ax=axes[1], color=["#4C78A8", "#F58518"])
axes[1].axhline(0.5, ls="--", color="gray"); axes[1].set_ylim(0, 1.05)
axes[1].set_title("Lo mismo, según el conector de la última opinión"); axes[1].set_xlabel("")
plt.tight_layout(); plt.show()
resumen_tipo.round(3)

**Qué sale:** las dos familias se comportan de forma opuesta.

- **Adjetivos:** la última opinión decide prácticamente siempre, con cualquier conector. Esta parte del conjunto de datos es **determinista**: un modelo que identifique bien la última opinión puede acertarla toda.
- **Frases hechas:** la etiqueta coincide con la última opinión en torno al 50 %, y el conector no cambia nada.

### 3.3 ¿Se puede predecir el grupo mixto de frases hechas?

Si existiera alguna regla oculta, un clasificador entrenado **solo** con esas reseñas debería superar el 50 %. Probamos TF‑IDF + regresión logística con validación cruzada de 5 *folds* y dos reglas simples.

In [ ]:
es_mixta = train_df["text"].map(es_mixta_frases) & (train_df["label"] != "neutral")
X_mix, y_mix = train_df.loc[es_mixta, "text"], train_df.loc[es_mixta, "label"]

def signo_primera_ultima(texto):
    s = [signo_frase(c) for c in separar_clausulas(texto) if signo_frase(c) != 0]
    return s[0], s[-1]
pu = np.array([signo_primera_ultima(t) for t in X_mix])
y_bin = np.where(y_mix == "positivo", 1, -1)

lr_mix = Pipeline([("tfidf", TfidfVectorizer(min_df=2, sublinear_tf=True, ngram_range=(1, 2), preprocessor=limpiar_texto)),
                   ("clf", LogisticRegression(C=3, max_iter=4000, random_state=RANDOM_STATE))])
acc_lr = cross_val_score(lr_mix, X_mix, y_mix, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)).mean()
pd.DataFrame({"accuracy": {"Clase mayoritaria": y_mix.value_counts(normalize=True).max(),
                           "Regla: gana la primera opinión": np.mean(pu[:, 0] == y_bin),
                           "Regla: gana la última opinión": np.mean(pu[:, 1] == y_bin),
                           "TF-IDF + regresión logística (CV 5 folds)": acc_lr}}).round(3).assign(reseñas=len(y_mix))

**Qué sale:** nada supera claramente el 50 %. Ni la posición de la opinión ni las palabras de la reseña predicen la etiqueta de este grupo, así que la tratamos como **ruido irreducible**. Probamos además otras variables (opinión más fuerte, suma de polaridades, conectores, distancia al final) y tampoco superaron el 50 %.

### 3.4 Techo estimado

Si el grupo mixto de frases hechas se acierta al 50 % y el resto se acierta casi perfecto, la accuracy máxima es aproximadamente `1 − 0.5 · (proporción del grupo)`. En `eval.csv` solo **contamos** el grupo con la misma función, que no usa etiquetas: no entrenamos ni elegimos nada con eval.

In [ ]:
frac_train = train_df["text"].map(es_mixta_frases).mean()
frac_eval = eval_df["text"].map(es_mixta_frases).mean()
techo = pd.DataFrame({"% reseñas mixtas con frases hechas": [frac_train, frac_eval],
                      "accuracy máxima aproximada": [1 - 0.5 * frac_train, 1 - 0.5 * frac_eval]},
                     index=["train.csv", "eval.csv"])
techo.round(3)

**Qué sale:** el grupo aleatorio es un poco mayor en eval que en train. El techo realista está alrededor de **0.92** de accuracy, y en la práctica algo menos, porque las listas de diagnóstico no detectan todas las reseñas de ese grupo. Esto explica por qué la entrega 5 y los otros grupos se estancan entre 0.89 y 0.92 en Kaggle. **El margen real de mejora está en la parte determinista.** Por eso, en la sección 7 medimos cada modelo por separado en cada grupo.

## 4. Modelos base de texto

Sin cambios respecto a la entrega 5. Ambos modelos reciben dos vistas de la reseña: el **texto completo** y la **última cláusula**. LR usa TF‑IDF con bigramas en el texto completo (`C=3`) y NB usa BoW binaria (`alpha=0.5`). Probamos agregar una vista de la penúltima cláusula y n‑gramas de caracteres, y ninguna de las dos mejoró en validación cruzada.

In [ ]:
def construir_vistas(textos):
    textos = pd.Series(textos).astype(str)
    return pd.DataFrame({"completo": textos.map(limpiar_texto).values,
                         "ultima_clausula": textos.map(lambda t: separar_clausulas(t)[-1]).values})

def vistas_texto(vectorizador_completo, vectorizador_ultima):
    return Pipeline([("vistas", FunctionTransformer(construir_vistas)),
                     ("vec", ColumnTransformer([("completo", vectorizador_completo, "completo"),
                                                ("ultima", vectorizador_ultima, "ultima_clausula")]))])

def modelo_lr(C=3):
    return Pipeline([("texto", vistas_texto(TfidfVectorizer(min_df=2, sublinear_tf=True, ngram_range=(1, 2)),
                                            TfidfVectorizer(min_df=2, sublinear_tf=True))),
                     ("clf", LogisticRegression(C=C, max_iter=4000, random_state=42))])

def modelo_nb(alpha=0.5):
    return Pipeline([("texto", vistas_texto(CountVectorizer(min_df=2, binary=True, ngram_range=(1, 2)),
                                            CountVectorizer(min_df=2, binary=True))),
                     ("clf", MultinomialNB(alpha=alpha))])

## 5. Variables de polaridad por cláusula

`PolaridadClausulas` es la clase de la entrega 5, sin cambios; la necesitamos para comparar. Los pesos de cada palabra son log‑cocientes de frecuencias de la bolsa de palabras: positivo frente a negativo para la **polaridad** y con opinión frente a neutral para la **opinión**. En entrenamiento se calculan *out-of-fold* para no filtrar la etiqueta.

`PolaridadClausulas2` cambia lo que mostró el análisis de errores de la entrega 5:

| Problema en la entrega 5 | Cambio |
|---|---|
| Modificadores de dos palabras: *poco fiable*, *mal terminada*, *de mala calidad* | El léxico de polaridad usa **unigramas y bigramas**. |
| Los cierres ambivalentes ensucian el léxico: se aprendían como última cláusula de reseñas positivas y negativas | El léxico se aprende con las últimas cláusulas que **no** son cierres (`CIERRES`). |
| La suma de pesos de una cláusula larga acumula ruido de las palabras de relleno | Variable **`pmax`**: peso de la palabra más polar de cada cláusula. |
| Relleno **después** de la opinión decisiva: *"…me resultó terrible ‖ lo usaba casi todos los días"* | Para κ = 2 y 3: polaridad, conector y distancia al final de la **última cláusula con una palabra fuertemente polar** (\|pmax\| > κ), además de la primera y el conteo. |
| — | Indicador de que la reseña termina en un cierre ambivalente. |

En total son 38 variables por reseña (21 en la entrega 5).

In [ ]:
class PolaridadClausulas(BaseEstimator, TransformerMixin):
    # Entrega 5 (sin cambios).
    def __init__(self, n_posiciones=4, n_folds=5, random_state=42):
        self.n_posiciones = n_posiciones; self.n_folds = n_folds; self.random_state = random_state

    @staticmethod
    def _log_ratio(vec, textos, ma, mb):
        X = vec.transform(textos)
        a = np.asarray(X[ma].sum(0)).ravel() + 1
        b = np.asarray(X[mb].sum(0)).ravel() + 1
        return np.log(a / a.sum()) - np.log(b / b.sum())

    def _aprender(self, cl, y):
        y = np.asarray(y)
        ultimas = [c[-1] for c in cl]; completos = [" ".join(c) for c in cl]
        vp = CountVectorizer(binary=True, min_df=2).fit(ultimas)
        vo = CountVectorizer(binary=True, min_df=2).fit(completos)
        return vp, self._log_ratio(vp, ultimas, y == "positivo", y == "negativo"), \
               vo, self._log_ratio(vo, completos, y != "neutral", y == "neutral")

    def _features(self, cl, pesos):
        vp, wp, vo, wo = pesos
        filas = []
        for c in cl:
            pol = vp.transform(c) @ wp; op = vo.transform(c) @ wo
            f = []
            for k in range(1, self.n_posiciones + 1):
                f += [pol[-k], op[-k], conector(c[-k])] if len(c) >= k else [0.0, -99.0, -2]
            j = np.where(op > 0)[0]
            f += [pol[j[-1]] if len(j) else 0.0, conector(c[j[-1]]) if len(j) else -2,
                  pol[j[-2]] if len(j) > 1 else 0.0, conector(c[j[-2]]) if len(j) > 1 else -2,
                  op.max(), op.sum(), pol.sum(), len(c), len(j)]
            filas.append(f)
        return np.array(filas, dtype=float)

    def nombres(self):
        n = []
        for k in range(self.n_posiciones):
            n += [f"pol_ult-{k}", f"op_ult-{k}", f"con_ult-{k}"]
        return n + ["pol_ult_opinion", "con_ult_opinion", "pol_penult_opinion", "con_penult_opinion",
                    "op_max", "op_total", "pol_total", "n_clausulas", "n_con_opinion"]

    def fit(self, X, y):
        self.pesos_ = self._aprender([separar_clausulas(t) for t in X], y); return self

    def transform(self, X):
        return self._features([separar_clausulas(t) for t in X], self.pesos_)

    def fit_transform(self, X, y):
        cl = [separar_clausulas(t) for t in X]; y = np.asarray(y)
        F = np.zeros((len(cl), len(self.nombres())))
        for i1, i2 in StratifiedKFold(self.n_folds, shuffle=True, random_state=self.random_state).split(cl, y):
            F[i2] = self._features([cl[i] for i in i2], self._aprender([cl[i] for i in i1], y[i1]))
        self.pesos_ = self._aprender(cl, y)
        return F


class PolaridadClausulas2(PolaridadClausulas):
    # Entrega 6: léxico con bigramas sin cierres, palabra más polar y última cláusula fuertemente polar.
    def __init__(self, n_posiciones=4, n_folds=5, random_state=42, kappas=(2.0, 3.0)):
        super().__init__(n_posiciones, n_folds, random_state)
        self.kappas = kappas

    def _aprender(self, cl, y):
        y = np.asarray(y)
        ultimas = [c[-1] for c in cl]; completos = [" ".join(c) for c in cl]
        ok = np.array([not es_cierre(u) for u in ultimas])   # los cierres ambivalentes no enseñan polaridad
        vp = CountVectorizer(binary=True, min_df=2, ngram_range=(1, 2)).fit([u for u, k in zip(ultimas, ok) if k])
        vo = CountVectorizer(binary=True, min_df=2).fit(completos)
        return vp, self._log_ratio(vp, ultimas, (y == "positivo") & ok, (y == "negativo") & ok), \
               vo, self._log_ratio(vo, completos, y != "neutral", y == "neutral")

    @staticmethod
    def _peso_maximo(Xc, wp):
        # Para cada cláusula (fila), el peso con mayor valor absoluto entre sus palabras.
        pmax = np.zeros(Xc.shape[0])
        for i in range(Xc.shape[0]):
            idx = Xc.indices[Xc.indptr[i]:Xc.indptr[i + 1]]
            if len(idx):
                pmax[i] = wp[idx[np.abs(wp[idx]).argmax()]]
        return pmax

    def _features(self, cl, pesos):
        vp, wp, vo, wo = pesos
        filas = []
        for c in cl:
            Xc = vp.transform(c); pol = Xc @ wp; op = vo.transform(c) @ wo
            pmax = self._peso_maximo(Xc, wp)
            f = []
            for k in range(1, self.n_posiciones + 1):
                f += [pol[-k], op[-k], pmax[-k], conector(c[-k])] if len(c) >= k else [0.0, -99.0, 0.0, -2]
            j = np.where(op > 0)[0]
            f += [pol[j[-1]] if len(j) else 0.0, conector(c[j[-1]]) if len(j) else -2,
                  pol[j[-2]] if len(j) > 1 else 0.0, conector(c[j[-2]]) if len(j) > 1 else -2]
            for kappa in self.kappas:
                jj = np.where(np.abs(pmax) > kappa)[0]
                f += [pol[jj[-1]] if len(jj) else 0.0, pmax[jj[-1]] if len(jj) else 0.0,
                      conector(c[jj[-1]]) if len(jj) else -2, len(c) - 1 - jj[-1] if len(jj) else -1,
                      pol[jj[0]] if len(jj) else 0.0, len(jj)]
            f += [op.max(), op.sum(), pol.sum(), len(c), len(j), float(es_cierre(c[-1]))]
            filas.append(f)
        return np.array(filas, dtype=float)

    def nombres(self):
        n = []
        for k in range(self.n_posiciones):
            n += [f"pol_ult-{k}", f"op_ult-{k}", f"pmax_ult-{k}", f"con_ult-{k}"]
        n += ["pol_ult_opinion", "con_ult_opinion", "pol_penult_opinion", "con_penult_opinion"]
        for kappa in self.kappas:
            s = f"{kappa:g}"
            n += [f"pol_fuerte{s}", f"pmax_fuerte{s}", f"con_fuerte{s}", f"dist_fuerte{s}",
                  f"pol_prim_fuerte{s}", f"n_fuerte{s}"]
        return n + ["op_max", "op_total", "pol_total", "n_clausulas", "n_con_opinion", "cierre_ambivalente"]


def modelo_rf(leaf=3):
    return Pipeline([("polaridad", PolaridadClausulas2(random_state=42)),
                     ("clf", RandomForestClassifier(n_estimators=500, min_samples_leaf=leaf, n_jobs=-1, random_state=42))])

### 5.1 Control: fuga de información en las variables nuevas

La distribución de la polaridad de la última cláusula fuertemente polar (`pol_fuerte2`) debe verse igual en train (*out-of-fold*) y en validación. Si en train las clases se separaran mucho mejor, habría fuga.

In [ ]:
pol2 = PolaridadClausulas2(random_state=RANDOM_STATE)
F_tr, F_va = pol2.fit_transform(X_train, y_train), pol2.transform(X_val)
col = pol2.nombres().index("pol_fuerte2")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharex=True, sharey=True)
for ax, F, y, t in [(axes[0], F_tr, y_train, "Train (out-of-fold)"), (axes[1], F_va, y_val, "Validación")]:
    for c in ORDER:
        ax.hist(F[np.asarray(y) == c, col], bins=50, range=(-20, 20), alpha=0.55, density=True, color=COLORS[c], label=c)
    ax.set_title(t); ax.set_xlabel("polaridad de la última cláusula fuertemente polar")
axes[0].legend(); plt.tight_layout(); plt.show()
print("Variables por reseña:", F_tr.shape[1])

**Qué sale:** las dos distribuciones son prácticamente iguales, así que no hay fuga. Las neutrales se concentran en 0 porque casi nunca tienen una palabra fuertemente polar. Positivas y negativas se separan hacia lados opuestos, con un traslape central que corresponde sobre todo a las reseñas mixtas.

### 5.2 Desempeño individual de los modelos base

Como referencia, medimos LR y NB sobre texto, y un Random Forest que usa **solo** las variables de `PolaridadClausulas2`.

In [ ]:
base = {"Regresión logística": modelo_lr(), "Naive Bayes": modelo_nb(), "Random Forest (polaridad 2)": modelo_rf()}
pred_base = {n: m.fit(X_train, y_train).predict(X_val) for n, m in base.items()}
res_base = pd.DataFrame({n: {"accuracy": accuracy_score(y_val, p), "F1 macro": f1_score(y_val, p, average="macro")}
                         for n, p in pred_base.items()}).T
res_base.round(4)

**Qué sale:** los tres modelos quedan alrededor de 0.85–0.91. Como en la entrega 5, cometen errores distintos, y eso es lo que aprovecha el ensamble.

## 6. Ensamble por apilamiento (*stacking*)

La lógica es la misma de la entrega 5:
1. Se obtienen con `cross_val_predict` (5 *folds*) las probabilidades **out-of-fold** de LR y NB.
2. Se calculan las variables de polaridad, también *out-of-fold*.
3. Se entrena un Random Forest meta (500 árboles, `min_samples_leaf=3`) sobre todo lo anterior.

`StackingPolaridad` es la versión de la entrega 5. Agregamos el método `_polaridad()` para que `StackingPolaridad2` solo cambie el transformador por `PolaridadClausulas2`. Así, cualquier diferencia entre las dos versiones viene únicamente de las variables y de la limpieza del texto.

In [ ]:
class StackingPolaridad(BaseEstimator, ClassifierMixin):
    # Nivel 1: LR (TF-IDF) y NB (BoW) -> probabilidades out-of-fold; variables de polaridad out-of-fold.
    # Nivel 2: Random Forest sobre [variables de polaridad + probabilidades LR + probabilidades NB].
    def __init__(self, C_lr=3, alpha_nb=0.5, leaf_meta=3, n_folds=5, random_state=42):
        self.C_lr = C_lr; self.alpha_nb = alpha_nb; self.leaf_meta = leaf_meta
        self.n_folds = n_folds; self.random_state = random_state

    def _polaridad(self):
        return PolaridadClausulas(n_folds=self.n_folds, random_state=self.random_state)

    def _meta_X(self, F, P_lr, P_nb):
        return np.hstack([F, P_lr, P_nb])

    def fit(self, X, y):
        X = pd.Series(X).reset_index(drop=True); y = np.asarray(y)
        cv = StratifiedKFold(self.n_folds, shuffle=True, random_state=self.random_state)
        self.lr_ = modelo_lr(self.C_lr); self.nb_ = modelo_nb(self.alpha_nb)
        P_lr = cross_val_predict(self.lr_, X, y, cv=cv, method="predict_proba")
        P_nb = cross_val_predict(self.nb_, X, y, cv=cv, method="predict_proba")
        self.pol_ = self._polaridad()
        F = self.pol_.fit_transform(X, y)
        self.lr_.fit(X, y); self.nb_.fit(X, y)
        self.X_meta_train_ = self._meta_X(F, P_lr, P_nb)
        self.meta_ = RandomForestClassifier(n_estimators=500, min_samples_leaf=self.leaf_meta, n_jobs=-1,
                                            random_state=self.random_state).fit(self.X_meta_train_, y)
        self.classes_ = self.meta_.classes_
        return self

    def meta_features(self, X):
        X = pd.Series(X)
        return self._meta_X(self.pol_.transform(X), self.lr_.predict_proba(X), self.nb_.predict_proba(X))

    def nombres_meta(self):
        return self.pol_.nombres() + [f"p_lr_{c}" for c in self.lr_.classes_] + [f"p_nb_{c}" for c in self.nb_.classes_]

    def predict_proba(self, X):
        return self.meta_.predict_proba(self.meta_features(X))

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(1)]


class StackingPolaridad2(StackingPolaridad):
    # Entrega 6: mismo ensamble con las variables de PolaridadClausulas2.
    def _polaridad(self):
        return PolaridadClausulas2(n_folds=self.n_folds, random_state=self.random_state)

In [ ]:
stack = StackingPolaridad2(random_state=RANDOM_STATE).fit(X_train, y_train)
pred_stack = stack.predict(X_val)
with limpieza_entrega5():
    pred_e5 = StackingPolaridad(random_state=RANDOM_STATE).fit(X_train, y_train).predict(X_val)
for nombre, p in [("Entrega 5", pred_e5), ("Entrega 6", pred_stack)]:
    print(f"{nombre} -> accuracy: {accuracy_score(y_val, p):.4f} | F1 macro: {f1_score(y_val, p, average='macro'):.4f}")

### 6.1 Control: complejidad del meta‑modelo

Variamos `min_samples_leaf` del Random Forest meta y comparamos el F1 en entrenamiento (sobre las variables *out-of-fold*) con el de validación. Mantenemos `min_samples_leaf=3`, que fue el mejor en la entrega 5 y en la validación cruzada. Esta curva solo verifica que el valor sigue siendo razonable.

In [ ]:
filas = []
for leaf in [1, 3, 5, 10, 20]:
    stack.meta_.set_params(min_samples_leaf=leaf).fit(stack.X_meta_train_, y_train)
    filas.append({"min_samples_leaf": leaf,
                  "F1 train (OOF)": f1_score(y_train, stack.meta_.predict(stack.X_meta_train_), average="macro"),
                  "F1 validación": f1_score(y_val, stack.predict(X_val), average="macro")})
res_leaf = pd.DataFrame(filas)
stack.meta_.set_params(min_samples_leaf=3).fit(stack.X_meta_train_, y_train)

ax = res_leaf.plot(x="min_samples_leaf", y=["F1 train (OOF)", "F1 validación"], marker="o", logx=True, figsize=(6, 3.5))
ax.set_title("Complejidad del meta-modelo (Random Forest)"); ax.set_ylabel("F1 macro")
plt.tight_layout(); plt.show()
res_leaf.round(4)

**Qué sale:** con hojas de 1 reseña el meta‑modelo memoriza (F1 de entrenamiento ≈ 1), y con hojas grandes se simplifica de más. Las diferencias en validación entre 1 y 10 son de décimas, del orden del ruido de una sola partición. `min_samples_leaf=3` sigue siendo una elección razonable.

## 7. Evaluación

### 7.1 Partición de validación (semilla 42)

In [ ]:
pred_stack = stack.predict(X_val)
print(f"Accuracy: {accuracy_score(y_val, pred_stack):.4f} | F1 macro: {f1_score(y_val, pred_stack, average='macro'):.4f}\n")
print(classification_report(y_val, pred_stack, digits=4))

resumen = pd.concat([res_base["F1 macro"], pd.Series({"Stacking entrega 5": f1_score(y_val, pred_e5, average="macro"),
                                                        "Stacking entrega 6": f1_score(y_val, pred_stack, average="macro")})])
fig, axes = plt.subplots(1, 3, figsize=(17, 4.3), gridspec_kw={"width_ratios": [1, 1.1, 1.3]})
ConfusionMatrixDisplay.from_predictions(y_val, pred_stack, labels=ORDER, cmap="Blues", ax=axes[0], colorbar=False)
axes[0].set_title("Matriz de confusión — stacking entrega 6")
resumen.plot(kind="barh", ax=axes[1], color=["#BAB0AC"] * 4 + ["#4C78A8"])
axes[1].set_xlim(0.8, 0.94); axes[1].set_title("F1 macro en validación (semilla 42)"); axes[1].invert_yaxis()
for i, v in enumerate(resumen.values):
    axes[1].text(v + 0.002, i, f"{v:.3f}", va="center")
imp = pd.Series(stack.meta_.feature_importances_, index=stack.nombres_meta()).sort_values().tail(15)
imp.plot(kind="barh", ax=axes[2], color="#4C78A8"); axes[2].set_title("Importancia en el meta-modelo (top 15)")
plt.tight_layout(); plt.show()

**Qué sale:** la clase `neutral` sigue casi perfecta y los errores se concentran entre `positivo` y `negativo`. En esta partición la diferencia entre las entregas 5 y 6 es pequeña, porque una sola partición de 2.400 reseñas incluye unas 360 del grupo aleatorio, cuyos aciertos cambian por azar. Para decidir usamos la validación cruzada de la sección 7.2.

### 7.2 Validación cruzada: entrega 5 frente a entrega 6

Obtenemos predicciones *out-of-fold* para las 12.000 reseñas de `train.csv`: cada reseña la predice un modelo que no la vio. Usamos las **mismas 5 particiones** para los dos modelos. La entrega 5 se evalúa con su limpieza original, sin colapsar letras.

Además de la accuracy y el F1 global, separamos el acierto en los grupos de la sección 3:
- **neutral**;
- **determinista**: reseñas no neutrales fuera del grupo mixto de frases hechas;
- **mixta con frases hechas**: grupo aleatorio.

El grupo determinista es donde una mejora real del modelo se tiene que notar. *(Esta celda tarda unos 7 minutos.)*

In [ ]:
folds_cv = list(StratifiedKFold(5, shuffle=True, random_state=0).split(train_df["text"], train_df["label"]))
X_all, y_all = train_df["text"], train_df["label"].values

def predicciones_oof(fabrica):
    pred = np.empty(len(y_all), dtype=object)
    for a, b in folds_cv:
        pred[b] = fabrica().fit(X_all.iloc[a], y_all[a]).predict(X_all.iloc[b])
    return pred

with limpieza_entrega5():
    oof_e5 = predicciones_oof(lambda: StackingPolaridad(random_state=RANDOM_STATE))
oof_e6 = predicciones_oof(lambda: StackingPolaridad2(random_state=RANDOM_STATE))

grupos = np.array([grupo(t, l) for t, l in zip(X_all, y_all)])
filas = {}
for nombre, p in [("Entrega 5", oof_e5), ("Entrega 6", oof_e6)]:
    fila = {"accuracy": accuracy_score(y_all, p), "F1 macro": f1_score(y_all, p, average="macro")}
    for g in ["neutral", "determinista", "mixta frases hechas"]:
        fila[f"acierto {g}"] = np.mean(p[grupos == g] == y_all[grupos == g])
    filas[nombre] = fila
res_cv = pd.DataFrame(filas).T
print("Reseñas por grupo:", pd.Series(grupos).value_counts().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
res_cv[["accuracy", "F1 macro"]].T.plot(kind="bar", rot=0, ax=axes[0], color=["#BAB0AC", "#4C78A8"], ylim=(0.88, 0.92))
axes[0].set_title("Validación cruzada (OOF, 5 folds)")
res_cv[[c for c in res_cv.columns if c.startswith("acierto")]].T.plot(kind="bar", rot=0, ax=axes[1], color=["#BAB0AC", "#4C78A8"], ylim=(0.4, 1.02))
axes[1].set_xticklabels(["neutral", "determinista", "mixta frases hechas"]); axes[1].axhline(0.5, ls="--", color="gray")
axes[1].set_title("Acierto por grupo de reseñas")
plt.tight_layout(); plt.show()
res_cv.round(4)

**Qué sale:**
- La entrega 6 mejora la accuracy y el F1 OOF. Lo que importa es que esa mejora viene del **grupo determinista**, que es justo donde el modelo puede mejorar de verdad.
- Las neutrales siguen casi perfectas en ambos modelos.
- En el grupo mixto de frases hechas los dos se quedan alrededor del 50 %, como se esperaba por la sección 3.
- Repetimos la comparación con otra semilla de particiones (`random_state=1`) mientras desarrollábamos la entrega y el resultado fue el mismo. En el grupo determinista, la entrega 5 dio 0.961 y la versión con las variables nuevas dio 0.965–0.966.

### 7.3 Ideas que probamos y descartamos

Todas se midieron con la misma validación cruzada:

| Idea | Resultado |
|---|---|
| Vista de la penúltima cláusula en LR | Empeora: el modelo lineal no sabe cuándo ignorarla. |
| n‑gramas de caracteres (`char_wb` 3–5) en LR, o como tercer modelo base | Sin mejora clara: empata y tarda más. |
| Seis posiciones en lugar de cuatro en las variables de polaridad | Sin mejora. |
| Random Forest meta con más árboles o `max_features=0.5` | Sin mejora. |
| Umbral fijo de "cláusula con opinión" o reglas de "última cláusula polar" | Peor que dejar que el Random Forest combine las variables. |
| Probabilidades de una regresión logística por cláusula como variables | Peor. |
| Quitar del entrenamiento las reseñas ambivalentes o las de etiqueta dudosa (OOF) | Sin mejora: su ruido no afecta la parte determinista. |

## 8. Modelo final, submission y guardado

Reentrenamos el *stacking* de la entrega 6 con **todo** `train.csv` y predecimos `eval.csv`. El modelo se guarda comprimido con `joblib`.

**Para cargar el `.joblib` en otro entorno** hay que ejecutar antes las celdas de las secciones 2, 4, 5 y 6, que definen las funciones y clases que usa el modelo. `NORMALIZAR_REPETIDAS` debe quedar en `True`.

In [ ]:
modelo_final = StackingPolaridad2(random_state=RANDOM_STATE).fit(train_df["text"], train_df["label"])
submission = pd.DataFrame({"id": eval_df["id"], "answer": modelo_final.predict(eval_df["text"])})
assert list(submission.columns) == list(sample_sub.columns) and len(submission) == len(sample_sub)
assert submission["id"].equals(sample_sub["id"]) and submission["answer"].isin(ORDER).all()
submission.to_csv("submission-G43-6.csv", index=False)

modelo_final.X_meta_train_ = None   # no se necesita para predecir; reduce el tamaño del archivo
joblib.dump(modelo_final, "models/modelo_entrega6_stacking.joblib", compress=3)
assert (joblib.load("models/modelo_entrega6_stacking.joblib").predict(eval_df["text"]) == submission["answer"]).all()
print(f"Tamaño del modelo: {Path('models/modelo_entrega6_stacking.joblib').stat().st_size / 1e6:.1f} MB")

ax = submission["answer"].value_counts(normalize=True).reindex(ORDER).plot(
    kind="bar", rot=0, figsize=(5, 3), color=[COLORS[c] for c in ORDER])
ax.set_title("Clases predichas en eval.csv"); plt.tight_layout(); plt.show()

**Qué sale:** la distribución predicha en `eval.csv` (≈ 35/30/35) coincide con la de entrenamiento. El modelo no colapsa hacia ninguna clase y se recarga desde disco con predicciones idénticas.

## 9. Conclusiones

| Entrega | Enfoque | F1 macro validación (semilla 42) |
|---|---|---|
| 1 | TF‑IDF + regresión logística | ≈ 0.75 |
| 2 | + TF‑IDF de la última oración | ≈ 0.89 |
| 3 | BoW binaria + Random Forest | ≈ 0.79 |
| 4 | Random Forest con polaridad por oración | ≈ 0.89 |
| 5 | Cláusulas + *stacking* (LR, NB + polaridad → RF) | ≈ 0.915 (OOF 5 folds: ≈ 0.908) |
| **6** | **Entrega 5 + normalización de tipeos + variables de polaridad 2** | ver secciones 7.1 y 7.2 |

1. **El techo lo pone el conjunto de datos.**
   - Alrededor del 15 % de las reseñas son mixtas con frases hechas, y su etiqueta no sigue ninguna regla que hayamos podido encontrar (≈ 50 % de acierto con cualquier método). Eso limita la accuracy a ≈ 0.92.
   - Explica por qué varias formas distintas de modelar (entregas 2, 4 y 5) se estancan en el mismo rango y por qué Kaggle queda por debajo de nuestra validación: `eval.csv` tiene una proporción algo mayor de ese grupo.
2. **La mejora de esta entrega es real pero acotada.**
   - La validación cruzada OOF sube en el grupo determinista, que es el único donde se podía ganar.
   - La ganancia global es de alrededor de medio punto. Es menor que la variación que puede tener el *leaderboard* público de Kaggle, que cambia más o menos un punto según qué reseñas aleatorias le toquen.
3. **Medir bien importó tanto como modelar.**
   - Una sola partición 80/20 sobreestimó la entrega 5 en casi un punto.
   - Separar el acierto por grupos permitió descartar ideas que parecían mejorar solo por azar.
4. **Límite de la Parte 1.** Lo que queda por ganar en la parte determinista son sobre todo errores de tipeo que la normalización no corrige (letras cambiadas de orden) y frases de opinión poco frecuentes. Los modelos secuenciales de la Parte 2 podrían capturar esto de forma más general, pero no el grupo aleatorio.